# Walkthrough: a solver through the typed `Task` ADT  ·  Colab edition

[\![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jdonaldson/turnstyle/blob/mc-selection-dispatch/experiments/dispatch_walkthrough_colab.ipynb)

Traces one prompt end-to-end through `DispatchTurnstyle` (the production consumer of
`turnstyle.dispatch`), using the richest path — a **snarks** `MultipleChoice` solver.

```
prompt → parse → MultipleChoice → enrich (selection shape) → solve (ChoiceProbe L13 argmax)
       → Answer("(B)") → SequenceLogitsProcessor → grounded output "(B)"
```

Self-contained: installs `turnstyle`, fetches the snarks data, and downloads SmolLM2-1.7B.
**Recommended: Runtime → Change runtime type → GPU** (the `fit_choice` probe sweep is much
faster on GPU; it works on CPU too, just slower).

## Setup — install, data, model

In [ ]:
# turnstyle (this branch) pulls torch / transformers / dateutil; Colab already has
# numpy + scikit-learn (used by autoprobe in fit_choice).
\!pip install -q "git+https://github.com/jdonaldson/turnstyle.git@mc-selection-dispatch" 

In [ ]:
# snarks straight from the BIG-Bench-Hard repo (same {input, target} the cache uses).
import requests
url = "https://raw.githubusercontent.com/suzgunmirac/BIG-Bench-Hard/main/bbh/snarks.json"
snarks = requests.get(url).json()["examples"]
print(len(snarks), "examples; target sample:", snarks[0]["target"])

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
tok = AutoTokenizer.from_pretrained(MODEL)
mdl = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=dtype).to(device).eval()
print("model on", device)

## The prompt
A sarcasm multiple-choice question — option (B) is the sarcastic one.

In [ ]:
prompt = snarks[0]["input"]
print(prompt)
print("\ngold:", snarks[0]["target"])

## Stage 1 — `parse`: raw prompt → a typed `Task`

`parse` is a cascade of **smart constructors** in priority order: each tries to claim the prompt
and returns `None` if it can't. For snarks, the value solvers decline (arithmetic is
**coverage-gated**), and the option markers `(A)/(B)` produce a `MultipleChoice`.

In [ ]:
from turnstyle.dispatch import parse, Ctx

ctx = Ctx(model=mdl, tokenizer=tok, device=device)
task = parse(prompt, ctx)
print("parse ->", task)

In [ ]:
# Why didn't arithmetic grab numbers in the prose? The coverage gate.
from turnstyle.arithmetic import parse_expression
from turnstyle.dispatch import _covers

res = parse_expression(prompt)
print("arithmetic sub-expression match:", res)
if res:
    print("covers the prompt? ", _covers(res[0], prompt), "  (a value solver must explain >= 50% of the prompt)")

## Stage 2 — `enrich`: model-based detectors fill the typed fields

`enrich` runs the one model forward pass: `detect_selection_shape` appends `"\nAnswer: ("`,
logit-lenses every layer to find the **decision layer**, and returns `PriorLocked` (committed
early, bias-driven) or `Deliberated(margin)` (committed late).

In [ ]:
from turnstyle.dispatch import enrich

task = enrich(task, prompt, ctx)
print("enriched ->", task)
print("selection shape:", task.selection)

## Stage 3 — fit the probe (one-time), then `solve`

`MultipleChoice` is the one variant that needs a model probe. `fit_choice` runs `autoprobe` over
the task's examples and attaches the fitted per-option `ProbeArtifact`. **A few min on GPU.**

In [ ]:
from turnstyle import DispatchTurnstyle

dt = DispatchTurnstyle(mdl, tok, device)
result = dt.fit_choice(snarks)          # autoprobe: layer x finder x mode, 5-fold CV
print(result.reason)

`solve` is the total `match` → `MultipleChoice -> _solve_choice`, which fires the **`ChoiceProbe`**:
it scores `P(sarcastic)` at **L13** for each option's last token and takes the **argmax**. This is
"interrupt argmax and solve" — read the model's good per-option scoring, do the comparison
externally, skip its weak stage-2 selection (the attention knockout proved that's causal).

In [ ]:
from turnstyle.dispatch import solve

ans = solve(task, prompt, dt.ctx)
print(ans)

## Stage 4 — grounding: the model is *made* to say it

The `Answer` becomes a `SequenceLogitsProcessor` that biases the model to emit `Answer.text` —
the probe's answer becomes the model's actual output.

In [ ]:
text, proof = dt.generate(prompt, max_new_tokens=8)
print("grounded output:", repr(text), " | gold:", snarks[0]["target"])

## Contrast — the skeleton without the model in the middle

A **deterministic** solver collapses stages 2–3: `parse` already knows the answer. An **abstain**
(no variant claims it) routes to `FreeForm -> Abstain` and falls back to plain generation.

In [ ]:
t, _ = dt.generate("What is 3 * (4 + 5)?", max_new_tokens=6)
print("arithmetic   ->", repr(t))

from turnstyle.dispatch import run, Abstain
r = run("Who composed the obscure opera nobody remembers?", ctx)
print("knowledge    ->", type(r).__name__, "| reason:", getattr(r, "reason", "-"))

## This one path is the whole project arc

- **Stage 1 coverage gate** — "commitment = coverage", the principle that replaced ad-hoc guards.
- **Stage 2 `detect_selection_shape`** — the decision-layer experiment, now a runtime detector.
- **Stage 3 `ChoiceProbe` @ L13 + external argmax** — "interrupt argmax and solve"; causally
  confirmed by the attention knockout.
- **Stage 4 `SequenceLogitsProcessor`** — neurosymbolic grounding.
- **Throughout** — `Task` sum type + `solve -> Answer | Abstain`, exhaustiveness compiler-checked.